In [35]:
require("readr")
require("ggplot2")
require("dplyr")
require("stringr")
require("lubridate")

Loading required package: lubridate


Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




In [83]:
costs <- read_csv("costs.csv")
gates <- read_csv("gates.csv")
transactions <- read_csv("transactions.csv")

Rows: 115204 Columns: 7
── Column specification ────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr  (5): Cost_ID, Cost_Type, Route_ID, From, To
dbl  (1): Amount
dttm (1): Cost_Date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 112 Columns: 5
── Column specification ────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (4): City, Gate_Number, Route_ID, Year
num (1): Annual_Gate_Cost

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 559985 Columns: 9
── Column specification ────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (9): Transaction_ID, Booking_ID, Transaction_Date, Route_ID, Transaction...

ℹ Use `spec()` to retriev

Cleaning costs dataset

In [77]:
costs <- costs |>
    mutate(Route_ID = toupper(Route_ID))
first <- str_extract(costs$Route_ID, "[A-Z]{3}")
second <- str_extract(costs$Route_ID, "[A-Z]{3}$")
combined <- paste0(first, '-', second)
costs$Route_ID = combined
costs

Cost_ID,Cost_Date,Cost_Type,Route_ID,From,To,Amount
<chr>,<dttm>,<chr>,<chr>,<chr>,<chr>,<dbl>
CO000001,2024-09-01 06:30:00,Crew,YVR-YLW,Vancouver,Kelowna,1388.53
CO000002,2024-09-01 06:30:00,Food,YVR-YLW,Vancouver,Kelowna,170.52
CO000003,2024-09-01 06:30:00,Fuel,YVR-YLW,Vancouver,Kelowna,1401.75
CO000004,2024-09-01 06:30:00,Maintenance,YVR-YLW,Vancouver,Kelowna,1657.45
CO000005,2024-09-01 06:35:00,Crew,YLW-YYC,Kelowna,Calgary,1673.28
CO000006,2024-09-01 06:35:00,Food,YLW-YYC,Kelowna,Calgary,221.60
CO000007,2024-09-01 06:35:00,Fuel,YLW-YYC,Kelowna,Calgary,1800.44
CO000008,2024-09-01 06:35:00,Maintenance,YLW-YYC,Kelowna,Calgary,1898.61
CO000009,2024-09-01 06:45:00,Crew,YYZ-YYC,Toronto,Calgary,12105.73


Cleaning gates dataset

In [28]:
gates <- gates |>
    mutate(Route_ID = toupper(Route_ID))
first <- str_extract(gates$Route_ID, "[A-Z]{3}")
second <- str_extract(gates$Route_ID, "[A-Z]{3}$")
combined <- paste0(first, '-', second)
gates$Route_ID = combined
gates

City,Gate_Number,Route_ID,Year,Annual_Gate_Cost
<chr>,<chr>,<chr>,<chr>,<dbl>
Abbotsford,G1,YXX-YYC,2024-2025,166800
Calgary,B1,YYC-YVR,2024-2025,373200
Calgary,B1,YYC-YYZ,2024-2025,373200
Calgary,B1,YYC-YEG,2024-2025,373200
Calgary,B2,YYC-YWG,2024-2025,351900
Calgary,B2,YYC-YXE,2024-2025,351900
Calgary,B2,YYC-YQR,2024-2025,351900
Calgary,B3,YYC-YLW,2024-2025,340700
Calgary,B3,YYC-YXX,2024-2025,340700


Cleaning transactions dataset

In [85]:
transactions <- transactions |>
    mutate(Route_ID = toupper(Route_ID)) |>
    mutate(Booking_Channel = toupper(Booking_Channel)) |>
    mutate(Transaction_Type = toupper(Transaction_Type)) |>
    mutate(Payment_Method = toupper(Payment_Method))
first <- str_extract(transactions$Route_ID, "[A-Z]{3}")
second <- str_extract(transactions$Route_ID, "[A-Z]{3}$")
combined <- paste0(first, '-', second)
transactions$Route_ID = combined

transactions$Transaction_Date <- format(
    parse_date_time(
          as.character(transactions$Transaction_Date),
          orders = c("ymd HM", "mdy HM", "dmy HM")
        ), 
    "%Y-%m-%d %H:%M")

transactions$Amount <- as.numeric(str_replace(str_replace(transactions$Amount, "\\$", ""), ",", ""))
transactions

Transaction_ID,Booking_ID,Transaction_Date,Route_ID,Transaction_Type,Membership_Tier,Booking_Channel,Payment_Method,Amount
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
TX1525667,BK396819,2026-07-23 00:00,YYZ-YHZ,TICKET,Non-Member,DIRECT WEB,CREDIT CARD,954.80
TX1420893,BK245942,2026-03-14 00:00,YVR-YEG,TICKET,Blue,CORPORATE PORTAL,CREDIT CARD,971.17
TX1313706,BK274024,2025-10-29 00:00,YVR-YEG,FOOD & ONBOARD,Non-Member,ONLINE TRAVEL AGENCY,CREDIT CARD,9.87
TX1169582,BK374084,2025-04-30 00:00,YYT-YHZ,TICKET,Silver,DIRECT WEB,CREDIT CARD,744.61
TX1378465,BK319834,2026-01-20 00:00,YYZ-YYC,TICKET,Blue,DIRECT WEB,CREDIT CARD,1380.38
TX1133037,BK382994,2025-03-11 00:00,YQB-YUL,EXTRA BAGGAGE,Blue,DIRECT WEB,CREDIT CARD,45.06
TX1112418,BK332533,2025-02-12 00:00,YHZ-YYZ,CHANGE FEE,Blue,DIRECT WEB,POINTS + CASH,75.00
TX1363471,BK329720,2026-01-01 00:00,YWG-YYC,TICKET,Non-Member,CORPORATE PORTAL,CREDIT CARD,1231.75
TX1413482,BK440357,2026-03-04 00:00,YYC-YVR,TICKET,Blue,MOBILE APP,CREDIT CARD,837.35
